<a href="https://colab.research.google.com/github/Neha-984055/Deep-Learning/blob/main/dl_skill_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

print("PyTorch Version:", torch.__version__)
print("Libraries imported successfully!")
print("Available device:", "GPU" if torch.cuda.is_available() else "CPU")


PyTorch Version: 2.14.1+cu130
Libraries imported successfully!
Available device: GPU


In [2]:

class SimpleModel(nn.Module):

    def __init__(self):
        super(SimpleModel, self).__init__()

        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)


model = SimpleModel()

print("Model Architecture:")
print(model)


Model Architecture:
SimpleModel(
  (network): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [3]:

# Create sample training data

X_train = torch.randn(100, 1, 28, 28)

# Create labels from 0 to 9
y_train = torch.randint(0, 10, (100,))

print("Training data shape:", X_train.shape)
print("Labels shape:", y_train.shape)
print("Sample data created successfully!")


Training data shape: torch.Size([100, 1, 28, 28])
Labels shape: torch.Size([100])
Sample data created successfully!


In [4]:

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

for epoch in range(5):

    optimizer.zero_grad()

    outputs = model(X_train)

    loss = criterion(outputs, y_train)

    loss.backward()

    optimizer.step()

    print("Epoch:", epoch + 1,
          "Loss:", loss.item())

print("Model training completed!")


Epoch: 1 Loss: 2.3214221000671387
Epoch: 2 Loss: 2.0528876781463623
Epoch: 3 Loss: 1.813061237335205
Epoch: 4 Loss: 1.5977686643600464
Epoch: 5 Loss: 1.4044103622436523
Model training completed!


In [5]:

checkpoint_path = "/content/model_checkpoint.pth"

torch.save(
    model.state_dict(),
    checkpoint_path
)

print("Checkpoint saved successfully!")
print("File:", checkpoint_path)


Checkpoint saved successfully!
File: /content/model_checkpoint.pth


In [6]:

loaded_model = SimpleModel()

loaded_model.load_state_dict(
    torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )
)

loaded_model.eval()

print("Checkpoint loaded successfully!")
print(loaded_model)


Checkpoint loaded successfully!
SimpleModel(
  (network): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [7]:

!pip install onnx onnxruntime


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 33.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 54.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [onnx]


In [8]:

import onnx

dummy_input = torch.randn(1, 1, 28, 28)

onnx_path = "/content/model.onnx"

torch.onnx.export(
    loaded_model,
    dummy_input,
    onnx_path,
    input_names=["input"],
    output_names=["output"],
    opset_version=17,
    dynamo=False
)

print("ONNX model exported successfully!")
print("ONNX File:", onnx_path)


ONNX model exported successfully!
ONNX File: /content/model.onnx


/tmp/ipykernel_488/2140673556.py:7: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


In [9]:

onnx_model = onnx.load(onnx_path)

onnx.checker.check_model(onnx_model)

print("ONNX Graph Validation Successful!")

print("Number of Nodes:",
      len(onnx_model.graph.node))

print("\nGraph Operations:")

for node in onnx_model.graph.node:
    print(node.op_type)


ONNX Graph Validation Successful!
Number of Nodes: 4

Graph Operations:
Flatten
Gemm
Relu
Gemm


In [10]:

import onnxruntime as ort

session = ort.InferenceSession(
    onnx_path,
    providers=["CPUExecutionProvider"]
)

test_input = np.random.randn(
    1, 1, 28, 28
).astype(np.float32)

result = session.run(
    None,
    {"input": test_input}
)

print("ONNX Runtime Test Successful!")
print("Output Shape:", result[0].shape)
print("Prediction Scores:", result[0])


ONNX Runtime Test Successful!
Output Shape: (1, 10)
Prediction Scores: [[ 0.41460252  0.38983744 -0.22622198 -0.39227682 -0.04858933 -0.57080543
  -0.670925    0.03821282  0.07605828  0.3387935 ]]


In [11]:

import importlib.util

print("GPU Available:",
      torch.cuda.is_available())

print("TensorRT Available:",
      importlib.util.find_spec("tensorrt") is not None)

if torch.cuda.is_available():
    print("GPU Name:",
          torch.cuda.get_device_name(0))
else:
    print("No NVIDIA GPU detected.")


GPU Available: True
TensorRT Available: False
GPU Name: Tesla T4


In [12]:

print("===== SKILL 7 MODEL ARTEFACT REPORT =====")

print("1. Model Architecture: Created")
print("2. Training: Completed")
print("3. Checkpoint: model_checkpoint.pth")
print("4. ONNX Graph: model.onnx")
print("5. ONNX Validation: Completed")

print("\nGenerated Files:")

for file in [
    "/content/model_checkpoint.pth",
    "/content/model.onnx"
]:
    print(file, "Exists:", os.path.exists(file))


===== SKILL 7 MODEL ARTEFACT REPORT =====
1. Model Architecture: Created
2. Training: Completed
3. Checkpoint: model_checkpoint.pth
4. ONNX Graph: model.onnx
5. ONNX Validation: Completed

Generated Files:
/content/model_checkpoint.pth Exists: True
/content/model.onnx Exists: True


In [15]:
from google.colab import files

In [16]:
files.download("/content/model.onnx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
files.download("/content/model_checkpoint.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>